# 3W Dataset: Transfer Learning

Measuring how well classifiers transfer across domains in the 3W dataset.
Cross-source (simulated/hand-drawn/real), mixed-source, and intra-real transfer
between different wells and time periods.

## 1. Setup and data loading

In [6]:
import os, glob, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import seaborn as sns
from scipy import stats
from scipy.stats import ks_2samp, wasserstein_distance
from sklearn.linear_model import SGDClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, accuracy_score
from sklearn.pipeline import Pipeline

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
np.random.seed(42)


In [7]:
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))

EXTRACTED_DATA_DIR = os.path.join(
    PROJECT_ROOT, 'data',
    'Data for A Realistic and Public Dataset with Rare Undesirable Real Events in Oil Wells',
    'data'
)

CLASS_NAMES = {
    0: 'Normal',
    1: 'Abrupt Increase of BSW',
    2: 'Spurious Closure of DHSV',
    3: 'Severe Slugging',
    4: 'Flow Instability',
    5: 'Rapid Productivity Loss',
    6: 'Quick Restriction in PCK',
    7: 'Scaling in PCK',
    8: 'Hydrate in Production Line',
}
VARIABLE_NAMES = ['P-PDG', 'P-TPT', 'T-TPT', 'P-MON-CKP',
                  'T-JUS-CKP', 'P-JUS-CKGL', 'T-JUS-CKGL', 'QGL']
SOURCES = ['real', 'simulated', 'handdrawn']


In [8]:
def detect_source(filename: str) -> str:
    name = os.path.splitext(filename)[0]
    if name.startswith('DRAWN'):   return 'handdrawn'
    if name.startswith('SIMULATED'): return 'simulated'
    if name.startswith('WELL'):    return 'real'
    return 'unknown'

def load_all_instances(data_dir: str, class_names: list) -> dict:
    all_instances = {}
    for class_code in class_names:
        class_dir = os.path.join(data_dir, str(class_code))
        if not os.path.isdir(class_dir):
            continue
        csv_files = sorted(glob.glob(os.path.join(class_dir, '*.csv')))
        if not csv_files:
            continue
        class_instances = {}
        for fpath in csv_files:
            fname = os.path.basename(fpath)
            source = detect_source(fname)
            df = pd.read_csv(fpath)
            if 'timestamp' in df.columns:
                df['timestamp'] = pd.to_datetime(df['timestamp'])
            instance_id = os.path.splitext(fname)[0]
            if source not in class_instances:
                class_instances[source] = []
            class_instances[source].append({
                'id': instance_id, 'path': fpath, 'data': df,
                'n_vars': len([c for c in df.columns if c not in ('timestamp', 'class')]),
                'n_obs': df.shape[0],
            })
        if class_instances:
            all_instances[class_code] = class_instances
    return all_instances

data = load_all_instances(EXTRACTED_DATA_DIR, CLASS_NAMES)

total = sum(sum(len(lst) for lst in src_dict.values()) for src_dict in data.values())
print(f'Loaded {total} instances across {len(data)} classes.')

Loaded 1984 instances across 9 classes.


In [9]:
source_pairs = [('real', 'simulated'), ('real', 'handdrawn'), ('simulated', 'handdrawn')]

def get_viable_classes(data, s1, s2, min_instances=3):
    """Return classes where both sources have >= min_instances instances."""
    return [cls for cls in sorted(data.keys())
            if s1 in data[cls] and s2 in data[cls]
            and len(data[cls][s1]) >= min_instances
            and len(data[cls][s2]) >= min_instances]


## 2. Transfer learning performance drop

In [10]:
def extract_features(data: dict, source: str, classes: list) -> tuple:
    """
    Extract statistical features per instance.
    Returns (X, y) where X is (n_instances, n_features) and y is class labels.

    Args:
        data (dict): The dataset dictionary.
        source (str): The source from which to extract features.
        classes (list): List of class codes to include in the feature extraction.
    Returns:
        tuple: (X, y, instance_ids) where X is the feature matrix, y is the array of class labels, and instance_ids is a list of instance identifiers.
    """
    features = []
    labels = []
    instance_ids = []
    
    for cls in classes:
        if cls not in data or source not in data[cls]:
            continue
        for inst in data[cls][source]:
            df = inst['data']
            feats = []
            for var in VARIABLE_NAMES:
                if var in df.columns:
                    vals = df[var].values
                    feats.extend([
                        np.nanmean(vals),
                        np.nanstd(vals),
                        np.nanmin(vals),
                        np.nanpercentile(vals, 25),
                        np.nanpercentile(vals, 50),
                        np.nanpercentile(vals, 75),
                        np.nanmax(vals),
                    ])
                else:
                    feats.extend([np.nan] * 7)
            
            features.append(feats)
            labels.append(cls)
            instance_ids.append(inst['id'])
    
    X = np.array(features, dtype=np.float64)
    y = np.array(labels)
    
    # nan values are replaced with column medians (or 0 if entire column is NaN)
    col_medians = np.nanmedian(X, axis=0)
    nan_mask = np.isnan(X)
    for j in range(X.shape[1]):
        if np.isnan(col_medians[j]):
            col_medians[j] = 0.0
        X[nan_mask[:, j], j] = col_medians[j]
    
    # replace inf values with column max/min (or 1/-1 if entire column is inf)
    for j in range(X.shape[1]):
        col = X[:, j]
        finite_vals = col[np.isfinite(col)]
        if len(finite_vals) > 0:
            col_max = np.max(finite_vals)
            col_min = np.min(finite_vals)
        else:
            col_max, col_min = 1.0, -1.0
        col[~np.isfinite(col) & (col > 0)] = col_max
        col[~np.isfinite(col) & (col < 0)] = col_min
        # clipping to avoid extreme values
        X[:, j] = np.clip(col, -1e15, 1e15)
    
    return X, y, instance_ids

stat_names = ['mean', 'std', 'min', 'p25', 'p50', 'p75', 'max']
feature_names = [f'{var}_{stat}' for var in VARIABLE_NAMES for stat in stat_names]
print(f'Feature vector: {len(feature_names)} dimensions')

Feature vector: 56 dimensions


In [11]:
def evaluate_transfer(data: dict, source: str, target: str, classes: list, n_trees: int = 100) -> dict | None:
    """
    Train on source, evaluate on target (and source for baseline).
    Performs multi-class classification on the given classes. Requires at least
    2 classes for meaningful evaluation.

    Args:
        data (dict): The dataset dictionary.
        source (str): The source domain to train on.
        target (str): The target domain to evaluate on.
        classes (list): List of class codes to include in the evaluation.
        n_trees (int): Number of trees for the Random Forest classifier.
    Returns:
        dict | None: Dictionary containing F1 scores and other metrics, or None if evaluation is not feasible due to insufficient data or classes.
    """
    X_src, y_src, _ = extract_features(data, source, classes)
    X_tgt, y_tgt, _ = extract_features(data, target, classes)
    
    if len(X_src) < 10 or len(X_tgt) < 5:
        return None
    if len(np.unique(y_src)) < 2 or len(np.unique(y_tgt)) < 2:
        return None
    
    if source == target:
        from sklearn.model_selection import train_test_split
        X_src, X_tgt, y_src, y_tgt = train_test_split(
            X_src, y_src, test_size=0.3, stratify=y_src, random_state=42)
    
    rf = RandomForestClassifier(n_estimators=n_trees, random_state=42, n_jobs=-1)
    
    # source to source: cross-validation to get baseline F1
    min_class_count = min(np.bincount(y_src))
    n_splits = max(2, min(5, min_class_count))
    try:
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
        y_pred_cv = cross_val_predict(rf, X_src, y_src, cv=skf)
        f1_src = f1_score(y_src, y_pred_cv, average='macro')
    except Exception:
        # fallback to a simple train/test split if cv fails
        from sklearn.model_selection import train_test_split
        X_tr, X_te, y_tr, y_te = train_test_split(
            X_src, y_src, test_size=0.3, stratify=y_src, random_state=42)
        rf.fit(X_tr, y_tr)
        f1_src = f1_score(y_te, rf.predict(X_te), average='macro')
    
    # source to target: train on all source, predict on target
    rf.fit(X_src, y_src)
    y_pred_tgt = rf.predict(X_tgt)
    f1_tgt = f1_score(y_tgt, y_pred_tgt, average='macro')
    acc_tgt = accuracy_score(y_tgt, y_pred_tgt)
    
    f1_drop = f1_src - f1_tgt
    
    return {
        'F1_source': f1_src,
        'F1_target': f1_tgt,
        'F1_drop': f1_drop,
        'accuracy_target': acc_tgt,
        'n_source': len(X_src),
        'n_target': len(X_tgt),
        'n_classes': len(np.unique(y_src)),
    }

transfer_scenarios = [
    ('real', 'real', 'Real → Real'),
    ('simulated', 'real', 'Simulated → Real'),
    ('handdrawn', 'real', 'Hand-drawn → Real'),
]

tl_results = []
for source, target, desc in transfer_scenarios:
    all_classes = get_viable_classes(data, source, target, min_instances=3)
    
    if len(all_classes) >= 2:
        result = evaluate_transfer(data, source, target, all_classes)
        if result:
            result['source'] = source
            result['target'] = target
            result['scenario'] = desc
            result['num_classes'] = len(all_classes)
            result['classes'] = ', '.join(str(c) for c in all_classes)
            tl_results.append(result)
    
    # evaluation per-class: leave-one-class-out within the source domain
    for cls in all_classes:
        other_classes = [c for c in all_classes if c != cls]
        if len(other_classes) >= 1:
            pair = [cls, other_classes[0]]  # binary: cls vs nearest other
            result = evaluate_transfer(data, source, target, pair)
            if result:
                result['source'] = source
                result['target'] = target
                result['scenario'] = desc
                result['class'] = cls
                result['event'] = CLASS_NAMES[cls][:25]
                tl_results.append(result)

tl_df = pd.DataFrame(tl_results)

multi_class = tl_df[~tl_df['class'].notna()] if 'class' in tl_df.columns else tl_df
if len(multi_class) > 0:
    print('\nMulti-class transfer')
    display(multi_class[['scenario', 'classes', 'num_classes', 'F1_source', 'F1_target',
                         'F1_drop', 'n_source', 'n_target']])

per_class = tl_df[tl_df['class'].notna()].copy() if 'class' in tl_df.columns else pd.DataFrame()
if len(per_class) > 0:
    print(f'\nPer-class binary transfer ({len(per_class)} results)')
    display(per_class[['scenario', 'class', 'event', 'F1_source', 'F1_target', 'F1_drop',
                       'n_source', 'n_target']].sort_values('F1_drop', ascending=False))



Multi-class transfer


,scenario,classes,num_classes,F1_source,F1_target,F1_drop,n_source,n_target
0,Real → Real,"0, 1, 2, 3, 4, 5, 6, 7, 8",9.0,0.606851,0.739258,-0.132407,717,308
10,Simulated → Real,"1, 2, 3, 5, 6, 8",6.0,0.983987,0.070902,0.913085,939,80
17,Hand-drawn → Real,"1, 7",2.0,0.900000,0.649351,0.250649,20,9



Per-class binary transfer (17 results)


,scenario,class,event,F1_source,F1_target,F1_drop,n_source,n_target
14,Simulated → Real,5.0,Rapid Productivity Loss,1.000000,0.000000,1.000000,553,17
13,Simulated → Real,3.0,Severe Slugging,0.988803,0.026316,0.962487,188,37
11,Simulated → Real,1.0,Abrupt Increase of BSW,1.000000,0.370370,0.629630,130,27
12,Simulated → Real,2.0,Spurious Closure of DHSV,1.000000,0.441379,0.558621,130,27
9,Real → Real,8.0,Hydrate in Production Lin,0.832736,0.498607,0.334129,420,180
19,Hand-drawn → Real,7.0,Scaling in PCK,0.949875,0.649351,0.300524,20,9
18,Hand-drawn → Real,1.0,Abrupt Increase of BSW,0.900000,0.649351,0.250649,20,9
7,Real → Real,6.0,Quick Restriction in PCK,0.698212,0.497222,0.200990,422,181
15,Simulated → Real,6.0,Quick Restriction in PCK,0.993316,0.803571,0.189744,329,11
16,Simulated → Real,8.0,Hydrate in Production Lin,1.000000,0.854545,0.145455,195,8


## 3. Mixed-source transfer

In [12]:
from sklearn.model_selection import train_test_split

def evaluate_mixed_transfer(data: dict, target: str, mix_source: str | None, classes: list, n_splits: int = 5, n_trees: int = 100) -> dict:
    """
    Evaluate whether mixing synthetic data with real data helps.
    
    Args:
        data (dict): The dataset dictionary.
        target (str): The target source (e.g., 'real').
        mix_source (str | None): The source to mix with the target (e.g., 'simulated' or 'handdrawn'). If None, only real data is used.
        classes (list): List of class codes to include in the evaluation.
        n_splits (int): Number of random splits for evaluation.
        n_trees (int): Number of trees for the Random Forest classifier.
    Returns:
        dict: Dictionary containing F1 scores and other metrics for real-only, mixed, and synthetic-only scenarios, as well as the number of training and testing instances.
    """
    X_tgt, y_tgt, ids_tgt = extract_features(data, target, classes)

    X_mix, y_mix = None, None
    if mix_source is not None:
        X_mix, y_mix, _ = extract_features(data, mix_source, classes)
    
    if len(X_tgt) < 6:  # need at least 3 train + 3 test
        return None
    
    f1_real_only = []   # real to real (within-domain)
    f1_mixed = []       # real + synthetic to real
    f1_synthetic_only = []  # synthetic to real (if applicable)

    rf = RandomForestClassifier(n_estimators=n_trees, random_state=42, n_jobs=-1)
    
    for seed in range(n_splits):
        try:
            X_tr, X_te, y_tr, y_te = train_test_split(
                X_tgt, y_tgt, test_size=0.5, stratify=y_tgt, random_state=seed)
        except ValueError:
            X_tr, X_te, y_tr, y_te = train_test_split(
                X_tgt, y_tgt, test_size=0.5, random_state=seed)
        
        rf.fit(X_tr, y_tr)
        f1_real_only.append(f1_score(y_te, rf.predict(X_te), average='macro'))
        
        if X_mix is not None and len(X_mix) > 0:
            X_combined = np.vstack([X_tr, X_mix])
            y_combined = np.concatenate([y_tr, y_mix])
            rf.fit(X_combined, y_combined)
            f1_mixed.append(f1_score(y_te, rf.predict(X_te), average='macro'))
        
        if X_mix is not None and len(X_mix) > 0:
            rf.fit(X_mix, y_mix)
            f1_synthetic_only.append(f1_score(y_te, rf.predict(X_te), average='macro'))
    
    result = {
        'F1_real_only_mean': np.mean(f1_real_only),
        'F1_real_only_std': np.std(f1_real_only),
        'F1_mixed_mean': np.nan,
        'F1_mixed_std': np.nan,
        'F1_synthetic_only_mean': np.nan,
        'F1_synthetic_only_std': np.nan,
        'delta_F1': np.nan,
        'n_train_real': len(X_tr),
        'n_test': len(X_te),
        'n_synthetic': len(X_mix) if X_mix is not None else 0,
    }
    if f1_mixed:
        result['F1_mixed_mean'] = np.mean(f1_mixed)
        result['F1_mixed_std'] = np.std(f1_mixed)
        result['delta_F1'] = np.mean(f1_mixed) - np.mean(f1_real_only)
    if f1_synthetic_only:
        result['F1_synthetic_only_mean'] = np.mean(f1_synthetic_only)
        result['F1_synthetic_only_std'] = np.std(f1_synthetic_only)
    return result

mixed_experiments = [
    ('real', None, 'Real only'),
    ('real', 'simulated', 'Real + Simulated'),
    ('real', 'handdrawn', 'Real + Hand-drawn'),
]

mixed_results = []
for target, mix_source, label in mixed_experiments:
    if mix_source is not None:
        classes = get_viable_classes(data, mix_source, target, min_instances=3)
        classes = [cls for cls in classes
                   if 'real' in data.get(cls, {})
                   and len(data[cls]['real']) >= 4]
    else:
        classes = [cls for cls in sorted(data.keys())
                   if 'real' in data.get(cls, {})
                   and len(data[cls]['real']) >= 6]
    
    if len(classes) < 2:
        continue
    
    result = evaluate_mixed_transfer(data, target, mix_source, classes)
    if result:
        result['experiment'] = label
        result['classes'] = ', '.join(str(c) for c in classes),
        result['n_classes'] = len(classes),
        mixed_results.append(result)

mixed_df = pd.DataFrame(mixed_results)
print(f'\nMixed-source transfer results ({len(mixed_df)} experiments):')
display(mixed_df[['experiment', 'n_classes', 'n_train_real', 'n_synthetic', 'n_test',
                  'F1_real_only_mean', 'F1_mixed_mean', 'F1_synthetic_only_mean', 'delta_F1']])


Mixed-source transfer results (3 experiments):


,experiment,n_classes,n_train_real,n_synthetic,n_test,F1_real_only_mean,F1_mixed_mean,F1_synthetic_only_mean,delta_F1
0,Real only,"(6,)",506,0,507,0.919381,NaN,NaN,NaN
1,Real + Simulated,"(5,)",38,858,39,0.894249,0.763673,0.103017,-0.130576
2,Real + Hand-drawn,"(2,)",4,20,5,0.568810,0.583333,0.583333,0.014524


## 4. Intra-real splits: Well-to-well transfer

Training on one set of wells and testing on another for the same event types.

In [13]:
def get_well_id(inst_id):
    if inst_id.startswith('WELL'): return inst_id.split('_')[0]
    return None

source_wells = ['WELL-00001', 'WELL-00002']
target_wells = ['WELL-00005', 'WELL-00006']

data_well = {}
for cls in sorted(data.keys()):
    data_well[cls] = {}
    src = [inst for inst in data[cls].get('real', []) if get_well_id(inst['id']) in source_wells]
    tgt = [inst for inst in data[cls].get('real', []) if get_well_id(inst['id']) in target_wells]
    if src: data_well[cls]['well_source'] = src
    if tgt: data_well[cls]['well_target'] = tgt

print('Well-based split instances per label:')
if 4 in data_well and 'well_source' in data_well[4] and 'well_target' in data_well[4]:
    print(f'\tClass 4 - well_source: {len(data_well[4]["well_source"])}')
    print(f'\tClass 4 - well_target: {len(data_well[4]["well_target"])}')
for k in ['well_source', 'well_target']:
    for c in sorted(data_well.keys()):
        if k in data_well[c]:
            print(f'\tClass {c} - {k}: {len(data_well[c][k])}')

Well-based split instances per label:
	Class 4 - well_source: 149
	Class 4 - well_target: 38
	Class 0 - well_source: 304
	Class 1 - well_source: 2
	Class 2 - well_source: 1
	Class 3 - well_source: 1
	Class 4 - well_source: 149
	Class 6 - well_source: 3
	Class 7 - well_source: 1
	Class 0 - well_target: 196
	Class 1 - well_target: 3
	Class 4 - well_target: 38
	Class 7 - well_target: 2


In [14]:
c4_src_wells = ['WELL-00001', 'WELL-00002']
c4_tgt_wells = ['WELL-00005', 'WELL-00010']

data_well_c4 = {}
for cls in [4]:
    data_well_c4[cls] = {}
    if cls in data and 'real' in data[cls]:
        data_well_c4[cls]['source'] = [inst for inst in data[cls]['real'] if get_well_id(inst['id']) in c4_src_wells]
        data_well_c4[cls]['target'] = [inst for inst in data[cls]['real'] if get_well_id(inst['id']) in c4_tgt_wells]

print('Class 4 (Flow instability) well split:')
print(f'\tSource wells 1+2: {len(data_well_c4[4]["source"])} instances')
print(f'\tTarget wells 5+10: {len(data_well_c4[4]["target"])} instances')

def collect_well_obs_c4(dw, wk, var):
    vals = []
    if 4 in dw and wk in dw[4]:
        for inst in dw[4][wk]:
            if var in inst['data'].columns:
                col = inst['data'][var].dropna().values[:20000]
                if len(col) > 0: vals.extend(col)
    return np.array(vals)

print('Well-to-well distances (class 4, Flow instability):')
for var in VARIABLE_NAMES:
    v1 = collect_well_obs_c4(data_well_c4, 'source', var)
    v2 = collect_well_obs_c4(data_well_c4, 'target', var)
    if len(v1) > 100 and len(v2) > 100:
        ks = ks_2samp(v1, v2).statistic
        pm, ps = np.mean(np.concatenate([v1, v2])), np.std(np.concatenate([v1, v2]))
        wd = wasserstein_distance((v1-pm)/ps, (v2-pm)/ps) if ps > 0 else 0
        print(f'\t{var}: KS={ks:.3f}, W={wd:.2f}')


Class 4 (Flow instability) well split:
	Source wells 1+2: 149 instances
	Target wells 5+10: 121 instances
Well-to-well distances (class 4, Flow instability):
	P-PDG: KS=0.576, W=1.17
	P-TPT: KS=0.439, W=0.99
	T-TPT: KS=1.000, W=1.67
	P-MON-CKP: KS=0.410, W=0.96
	T-JUS-CKP: KS=0.991, W=1.66
	QGL: KS=1.000, W=2.07


In [15]:
# Well-to-well transfer performance
well_cls_multi = [c for c in sorted(data.keys())
                  if 'well_source' in data_well.get(c, {})
                  and 'well_target' in data_well.get(c, {})
                  and len(data_well[c]['well_source']) >= 3
                  and len(data_well[c]['well_target']) >= 3]

print(f'Viable classes for well-to-well TL: {well_cls_multi}')

if len(well_cls_multi) >= 2:
    X_src, y_src, _ = extract_features(data_well, 'well_source', well_cls_multi)
    X_tgt, y_tgt, _ = extract_features(data_well, 'well_target', well_cls_multi)
    for X in [X_src, X_tgt]:
        cm = np.nanmedian(X, axis=0)
        for j in range(X.shape[1]):
            if np.isnan(cm[j]): cm[j] = 0.0
            mask = np.isnan(X[:, j])
            if mask.any(): X[mask, j] = cm[j]
            X[:, j] = np.clip(X[:, j], -1e15, 1e15)
    rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    min_cls = min(np.bincount(y_src))
    n_splits = max(2, min(5, min_cls))
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    y_cv = cross_val_predict(rf, X_src, y_src, cv=skf)
    f1_src = f1_score(y_src, y_cv, average='macro')
    rf.fit(X_src, y_src)
    f1_tgt = f1_score(y_tgt, rf.predict(X_tgt), average='macro')
    print(f'\tWells 1+2 CV F1={f1_src:.3f} -> Wells 5+6 F1={f1_tgt:.3f} (drop={f1_src-f1_tgt:.3f})')

# Well-to-well transfer for class 4 (Flow instability) with multi-class [0, 4]
c4_tgt_wells_full = ['WELL-00005', 'WELL-00010']
data_well_c4_multi = {}
for cls in [0, 4]:
    data_well_c4_multi[cls] = {}
    src = [inst for inst in data[cls].get('real', []) if get_well_id(inst['id']) in c4_src_wells]
    tgt = [inst for inst in data[cls].get('real', []) if get_well_id(inst['id']) in c4_tgt_wells_full]
    if src: data_well_c4_multi[cls]['source'] = src
    if tgt: data_well_c4_multi[cls]['target'] = tgt

c4_viable = [c for c in [0, 4] if 'source' in data_well_c4_multi.get(c, {}) and 'target' in data_well_c4_multi.get(c, {})]
print(f'Class 4 well-to-well multi-class TL (classes {c4_viable}):')
print(f'  Source wells {c4_src_wells}: {sum(len(data_well_c4_multi[c].get("source", [])) for c in c4_viable)} instances')
print(f'  Target wells {c4_tgt_wells_full}: {sum(len(data_well_c4_multi[c].get("target", [])) for c in c4_viable)} instances')

if len(c4_viable) >= 2:
    X4_src, y4_src, _ = extract_features(data_well_c4_multi, 'source', c4_viable)
    X4_tgt, y4_tgt, _ = extract_features(data_well_c4_multi, 'target', c4_viable)
    for X in [X4_src, X4_tgt]:
        cm = np.nanmedian(X, axis=0)
        for j in range(X.shape[1]):
            if np.isnan(cm[j]): cm[j] = 0.0
            mask = np.isnan(X[:, j])
            if mask.any(): X[mask, j] = cm[j]
            X[:, j] = np.clip(X[:, j], -1e15, 1e15)
    rf4 = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    min_cls4 = min(np.bincount(y4_src))
    n_splits4 = max(2, min(5, min_cls4))
    skf4 = StratifiedKFold(n_splits=n_splits4, shuffle=True, random_state=42)
    y4_cv = cross_val_predict(rf4, X4_src, y4_src, cv=skf4)
    f1_4_src = f1_score(y4_src, y4_cv, average='macro')
    rf4.fit(X4_src, y4_src)
    f1_4_tgt = f1_score(y4_tgt, rf4.predict(X4_tgt), average='macro')
    print(f'  Source CV F1={f1_4_src:.3f} -> Target F1={f1_4_tgt:.3f} (drop={f1_4_src-f1_4_tgt:.3f})')

Viable classes for well-to-well TL: [0, 4]
	Wells 1+2 CV F1=0.983 -> Wells 5+6 F1=0.869 (drop=0.114)
Class 4 well-to-well multi-class TL (classes [0, 4]):
  Source wells ['WELL-00001', 'WELL-00002']: 453 instances
  Target wells ['WELL-00005', 'WELL-00010']: 202 instances
  Source CV F1=0.983 -> Target F1=0.880 (drop=0.102)


## 5. Intra-real splits: Time-period transfer

Testing concept drift: training on old data, testing on newer data from the same well.

In [16]:
def get_year(inst_id: str) -> int | None:
    """
    Extract the year from an instance ID if it follows the expected format.

    Args:
        inst_id (str): The instance ID string.
    Returns:
        int | None: The extracted year as an integer, or None if the format is invalid.
    """
    parts = inst_id.split('_')
    if len(parts) >= 2 and parts[0].startswith('WELL'):
        ts = parts[-1]
        if len(ts) >= 8 and ts.isdigit(): return int(ts[:4])
    return None

CLS = 0
WID = 'WELL-00002'

inst_by_year = {}
if CLS in data and 'real' in data[CLS]:
    for inst in data[CLS]['real']:
        if get_well_id(inst['id']) == WID:
            y = get_year(inst['id'])
            if y: inst_by_year.setdefault(y, []).append(inst)

print(f'Class 0, {WID}: instances per year')
for y in sorted(inst_by_year):
    print(f'  {y}: {len(inst_by_year[y])}')

time_src = inst_by_year.get(2013, [])
time_tgt = inst_by_year.get(2017, [])
print(f'\nSource (2013): {len(time_src)}, Target (2017): {len(time_tgt)}')

if 4 in data and 'real' in data[4]:
    inst_by_year_4 = {}
    for inst in data[4]['real']:
        if get_well_id(inst['id']) == WID:
            y = get_year(inst['id'])
            if y: inst_by_year_4.setdefault(y, []).append(inst)
    print(f'Class 4, {WID}: instances per year')
    for y in sorted(inst_by_year_4):
        print(f'  {y}: {len(inst_by_year_4[y])}')
    time_src_4 = inst_by_year_4.get(2013, [])
    time_tgt_4 = inst_by_year_4.get(2014, [])
    print(f'\nClass 4 Source (2013): {len(time_src_4)}, Target (2014): {len(time_tgt_4)}')


Class 0, WELL-00002: instances per year
  2013: 8
  2017: 202

Source (2013): 8, Target (2017): 202
Class 4, WELL-00002: instances per year
  2013: 23
  2014: 90

Class 4 Source (2013): 23, Target (2014): 90


In [17]:
# Time-period PAD (class 0)
if len(time_src) >= 3 and len(time_tgt) >= 3:
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.linear_model import SGDClassifier
    from sklearn.model_selection import StratifiedKFold
    from sklearn.metrics import accuracy_score
    
    def extract_features_simple(instances):
        feats = []
        for inst in instances:
            f = []
            for var in VARIABLE_NAMES:
                if var in inst['data'].columns:
                    v = inst['data'][var].dropna().values
                    f += [np.nanmean(v), np.nanstd(v), np.nanpercentile(v, 50)]
                else: f += [np.nan]*3
            feats.append(f)
        return np.array(feats)
    
    X_s = extract_features_simple(time_src)
    X_t = extract_features_simple(time_tgt)
    for X in [X_s, X_t]:
        cm = np.nanmedian(X, axis=0)
        for j in range(X.shape[1]):
            if np.isnan(cm[j]): cm[j] = 0.0
            m = np.isnan(X[:, j])
            if m.any(): X[m, j] = cm[j]
    
    X_all = np.vstack([X_s, X_t])
    y_all = np.array([0]*len(X_s) + [1]*len(X_t))
    clf = Pipeline([('scaler', StandardScaler()),
                    ('sgd', SGDClassifier(loss='log_loss', max_iter=1000, tol=1e-3, random_state=42))])
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    errs = []
    for tr, te in skf.split(X_all, y_all):
        clf.fit(X_all[tr], y_all[tr])
        errs.append(1 - accuracy_score(y_all[te], clf.predict(X_all[te])))
    me = np.mean(errs)
    pad = max(0, min(1, 2*(1-2*me)))
    print(f'Class 0 time PAD = {pad:.3f}, n_src={len(X_s)}, n_tgt={len(X_t)}')


Class 0 time PAD = 1.000, n_src=8, n_tgt=202


In [18]:
# Time-period TL: train 2013, test 2017 (class 0)
if len(time_src) >= 4 and len(time_tgt) >= 4:
    from sklearn.model_selection import train_test_split
    
    def build_time_data(data, cls, well_id, year_src, year_tgt):
        td = {cls: {}}
        td[cls]['src'] = []
        td[cls]['tgt'] = []
        for inst in data.get(cls, {}).get('real', []):
            if get_well_id(inst['id']) == well_id:
                y = get_year(inst['id'])
                if y == year_src: td[cls]['src'].append(inst)
                if y == year_tgt: td[cls]['tgt'].append(inst)
        return td
    
    td_tl = build_time_data(data, 0, 'WELL-00002', 2013, 2017)
    if len(td_tl[0]['src']) >= 4 and len(td_tl[0]['tgt']) >= 4:
        X_src0, y_src0, _ = extract_features(td_tl, 'src', [0])
        X_tgt0, y_tgt0, _ = extract_features(td_tl, 'tgt', [0])
        if len(np.unique(y_src0)) >= 2 and len(np.unique(y_tgt0)) >= 2:
            rf = RandomForestClassifier(n_estimators=100, random_state=42)
            rf.fit(X_src0, y_src0)
            f1_s = f1_score(y_src0, rf.predict(X_src0), average='macro')
            f1_t = f1_score(y_tgt0, rf.predict(X_tgt0), average='macro')
            print(f'Class 0 time TL: train F1={f1_s:.3f}, test F1={f1_t:.3f}, drop={f1_s-f1_t:.3f}')


## 6. Key Findings

### Cross-source transfer
- Simulated -> Real: F1 drops 0.98 (within-source CV) to 0.07 (target)
- Hand-drawn -> Real: F1 drops 0.90 to 0.65 (moderate, 3.6x better than 939 simulated)
- Per-class: C5 (Prod. Loss) and C3 (Slugging) have the worst simulated-to-real transfer

### Mixed-source
- Adding simulated to real training degrades performance (F1 drop -0.13 vs real-only)
- Adding hand-drawn is neutral to slightly positive (+0.01)
- Quantity does not compensate for quality in synthetic data

### Well-to-well transfer
- Class 0 (Normal): measurable drop (F1 0.98 -> 0.87) across different well groups
- Class 4 (Flow Instability): wells are perfectly separable (PAD=1.0) for the same fault

### Time-period transfer
- Class 0 (2013 -> 2017): PAD=1.0, clear concept drift over 4 years
- Variables like P-TPT, P-MON-CKP show complete CDF separation; QGL shows zero drift
- Domain adaptation should focus on drift-prone sensors
